# HWI, transport--entropy, and entropy decay along the OU flow

This notebook generates `fig:gradflow-hwi-entropy-decay`.  The reference measure is the standard Gaussian
$\gamma$, and the initial density is a non-Gaussian one-dimensional Gaussian mixture.  The Ornstein--Uhlenbeck flow has the explicit law
$$
    X_t=e^{-t}X_0+\sqrt{1-e^{-2t}}Z,
    \qquad Z\sim\mathcal N(0,1),
$$
so the evolving density remains a Gaussian mixture.  On a fine grid we compute
$$
H(t)=\mathrm{KL}(\alpha_t|\gamma),\qquad
I(t)=\int \left|\nabla\log\frac{\rho_t}{\gamma}\right|^2\rho_t,\qquad
W(t)=W_2(\alpha_t,\gamma),
$$
and compare the Talagrand, HWI, logarithmic Sobolev, and entropy-decay estimates.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
from IPython.display import Image, display
from scipy.special import erfinv

ROOT = Path.cwd()
if ROOT.name == "notebooks-figures":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "notebooks-figures"))

from figure_style import BLUE, RED, VIOLET, GRAY, box_axes, figure_dir, interp_color, save_pdf, setup_matplotlib

setup_matplotlib()
OUT = figure_dir("gradflow-hwi-entropy-decay")
PNG_OUT = ROOT / "notebooks-figures" / "thumbnails"
PNG_OUT.mkdir(exist_ok=True)

## Explicit mixture evolution and grid quadrature

The displayed initial density is deliberately asymmetric and bimodal, so the relaxation is visible.  The grid is wide enough that all reported masses and moments are stable; the Wasserstein distance is computed by exact one-dimensional quantile inversion on the numerical CDF.

In [ ]:
weights = np.array([0.60, 0.40])
means0 = np.array([-1.62, 1.08])
sigmas0 = np.array([0.46, 0.64])

x = np.linspace(-7.5, 7.5, 9001)
dx = x[1] - x[0]
u = np.linspace(2e-5, 1 - 2e-5, 5000)
q_gamma = np.sqrt(2.0) * erfinv(2.0 * u - 1.0)

def gaussian_pdf(x, m, s):
    z = (x - m) / s
    return np.exp(-0.5 * z**2) / (np.sqrt(2 * np.pi) * s)

def gaussian_pdf_derivative(x, m, s):
    g = gaussian_pdf(x, m, s)
    return -((x - m) / (s**2)) * g

gamma = gaussian_pdf(x, 0.0, 1.0)

def density_and_derivative(t):
    a = np.exp(-t)
    means = a * means0
    sigmas = np.sqrt(a**2 * sigmas0**2 + (1.0 - a**2))
    rho = np.zeros_like(x)
    rho_prime = np.zeros_like(x)
    for w, m, s in zip(weights, means, sigmas):
        rho += w * gaussian_pdf(x, m, s)
        rho_prime += w * gaussian_pdf_derivative(x, m, s)
    rho /= np.trapezoid(rho, x)
    rho_prime /= np.trapezoid(rho, x)
    return rho, rho_prime

def quantile_from_density(rho):
    cdf = np.cumsum((rho[:-1] + rho[1:]) * 0.5 * dx)
    cdf = np.concatenate([[0.0], cdf])
    cdf /= cdf[-1]
    return np.interp(u, cdf, x)

def metrics(t):
    rho, rho_prime = density_and_derivative(t)
    eps = 1e-300
    score_ratio = rho_prime / np.maximum(rho, eps) + x
    H = np.trapezoid(rho * (np.log(np.maximum(rho, eps)) - np.log(gamma)), x)
    I = np.trapezoid(score_ratio**2 * rho, x)
    q = quantile_from_density(rho)
    W2 = np.trapezoid((q - q_gamma) ** 2, u)
    return H, I, W2, rho

times_curve = np.linspace(0, 3.0, 181)
H = np.zeros_like(times_curve)
I = np.zeros_like(times_curve)
W2 = np.zeros_like(times_curve)
densities = []
for k, t in enumerate(times_curve):
    H[k], I[k], W2[k], rho = metrics(float(t))
    if k in [0, 9, 24, 55, 120, 180]:
        densities.append((float(t), rho))

HWI = np.sqrt(np.maximum(W2, 0.0)) * np.sqrt(np.maximum(I, 0.0)) - 0.5 * W2
LSI = 0.5 * I
T2 = 0.5 * W2
entropy_envelope = H[0] * np.exp(-2 * times_curve)

print("max H-HWI", np.max(H - HWI))
print("max H-LSI", np.max(H - LSI))
print("max T2-H", np.max(T2 - H))
print("max H-envelope", np.max(H - entropy_envelope))

## Exported panels

The density panel gives the intuitive evolution.  The inequality panel plots the three static estimates on the same axes.  The decay panel isolates the dynamic consequence of the log-Sobolev inequality.

In [ ]:
def plot_density_panel():
    fig, ax = plt.subplots(figsize=(2.62, 1.78))
    for k, (t, rho) in enumerate(densities):
        color = interp_color(k / (len(densities) - 1))
        ax.plot(x, rho, color=color, lw=1.08)
        ax.fill_between(x, 0, rho, color=color, alpha=0.045, linewidth=0)
    ax.plot(x, gamma, color="#111111", lw=0.85, alpha=0.55, ls="--")
    ax.set_xlim(-4.0, 4.0)
    ax.set_ylim(0, 1.08 * max(rho.max() for _, rho in densities))
    ax.set_xlabel(r"$x$", labelpad=1.5)
    ax.set_ylabel(r"density", labelpad=2.0)
    ax.tick_params(labelsize=7, pad=1.5)
    box_axes(ax)
    save_pdf(fig, OUT / "ou-densities.pdf", pad_inches=0.035)
    return fig

fig = plot_density_panel()
plt.close(fig)

fig, ax = plt.subplots(figsize=(2.80, 1.78))
ax.semilogy(times_curve, H, color=VIOLET, lw=1.55, label=r"$H$")
ax.semilogy(times_curve, HWI, color=BLUE, lw=1.05, label=r"HWI upper")
ax.semilogy(times_curve, LSI, color=GRAY, lw=1.0, ls="--", label=r"LSI upper")
ax.semilogy(times_curve, T2, color=RED, lw=1.0, ls=":", label=r"$T_2$ lower")
ax.set_xlim(0, 3.0)
positive = np.concatenate([H, HWI, LSI, T2])
positive = positive[positive > 1e-8]
ax.set_ylim(max(positive.min() * 0.8, 1e-5), positive.max() * 1.25)
ax.set_xlabel(r"$t$", labelpad=1.5)
ax.set_ylabel(r"quantity", labelpad=2.0)
ax.tick_params(labelsize=7, pad=1.5)
ax.legend(frameon=False, fontsize=6.8, loc="upper right", handlelength=1.8, borderpad=0.1, labelspacing=0.17)
box_axes(ax)
save_pdf(fig, OUT / "static-bounds.pdf", pad_inches=0.035)
plt.close(fig)

fig, ax = plt.subplots(figsize=(2.46, 1.78))
ax.semilogy(times_curve, H, color=VIOLET, lw=1.55, label=r"$H(t)$")
ax.semilogy(times_curve, entropy_envelope, color=GRAY, lw=1.05, ls="--", label=r"$H(0)e^{-2t}$")
for idx in [0, 9, 24, 55, 120, 180]:
    ax.scatter(times_curve[idx], H[idx], s=12, color=interp_color(idx / (len(times_curve) - 1)), zorder=4)
ax.set_xlim(0, 3.0)
ax.set_ylim(max(H[-1] * 0.75, 1e-4), H[0] * 1.15)
ax.set_xlabel(r"$t$", labelpad=1.5)
ax.set_ylabel(r"entropy", labelpad=2.0)
ax.tick_params(labelsize=7, pad=1.5)
ax.legend(frameon=False, fontsize=7, loc="upper right", handlelength=1.8, borderpad=0.1, labelspacing=0.15)
box_axes(ax)
save_pdf(fig, OUT / "entropy-decay.pdf", pad_inches=0.035)
plt.close(fig)

# Compact PNG thumbnail matching the LaTeX layout.
thumb, axs = plt.subplots(1, 3, figsize=(8.1, 1.82), gridspec_kw={"width_ratios": [1.05, 1.15, 1.0], "wspace": 0.28})
ax = axs[0]
for k, (t, rho) in enumerate(densities):
    color = interp_color(k / (len(densities) - 1))
    ax.plot(x, rho, color=color, lw=1.0)
    ax.fill_between(x, 0, rho, color=color, alpha=0.040, linewidth=0)
ax.plot(x, gamma, color="#111111", lw=0.78, alpha=0.55, ls="--")
ax.set_xlim(-4.0, 4.0)
ax.set_ylim(0, 1.08 * max(rho.max() for _, rho in densities))
ax.set_xlabel(r"$x$", labelpad=1.0)
ax.set_ylabel("density", labelpad=1.0)
ax.tick_params(labelsize=6.5, pad=1.0)
box_axes(ax)
ax = axs[1]
ax.semilogy(times_curve, H, color=VIOLET, lw=1.38)
ax.semilogy(times_curve, HWI, color=BLUE, lw=0.95)
ax.semilogy(times_curve, LSI, color=GRAY, lw=0.9, ls="--")
ax.semilogy(times_curve, T2, color=RED, lw=0.9, ls=":")
ax.set_xlim(0, 3.0)
ax.set_ylim(max(positive.min() * 0.8, 1e-5), positive.max() * 1.25)
ax.set_xlabel(r"$t$", labelpad=1.0)
ax.set_ylabel("quantity", labelpad=1.0)
ax.tick_params(labelsize=6.5, pad=1.0)
box_axes(ax)
ax = axs[2]
ax.semilogy(times_curve, H, color=VIOLET, lw=1.38)
ax.semilogy(times_curve, entropy_envelope, color=GRAY, lw=0.95, ls="--")
for idx in [0, 9, 24, 55, 120, 180]:
    ax.scatter(times_curve[idx], H[idx], s=9, color=interp_color(idx / (len(times_curve) - 1)), zorder=4)
ax.set_xlim(0, 3.0)
ax.set_ylim(max(H[-1] * 0.75, 1e-4), H[0] * 1.15)
ax.set_xlabel(r"$t$", labelpad=1.0)
ax.set_ylabel("entropy", labelpad=1.0)
ax.tick_params(labelsize=6.5, pad=1.0)
box_axes(ax)
thumb.savefig(PNG_OUT / "gradflow-hwi-entropy-decay.png", dpi=190, bbox_inches="tight", pad_inches=0.025)
plt.close(thumb)

print(f"saved panels in {OUT.relative_to(ROOT)}")

## Figure preview

The preview shows the density evolution; the LaTeX figure combines it with the static-bounds and entropy-decay panels.

In [ ]:
display(Image(filename=str(PNG_OUT / "gradflow-hwi-entropy-decay.png")))